# 03 — Baseline ResNet-50 V2 no Kaggle Notebooks (primário, ADR-0009)

Treino do baseline ResNet-50 conforme:
- [ADR-0008](https://github.com/ThiagoCB1900/TCC/blob/main/docs/decisions/0008-framework-treino-pytorch-puro.md) (PyTorch puro)
- [ADR-0007](https://github.com/ThiagoCB1900/TCC/blob/main/docs/decisions/0007-tratamento-de-desbalanceamento.md) (weighted CE balanced + ablação sem peso)
- **[ADR-0010](https://github.com/ThiagoCB1900/TCC/blob/main/docs/decisions/0010-baseline-v2-hiperparametros-anti-overfit.md) — V2**: hiperparâmetros revisitados após F-0013 (overfit catastrófico em 4 epochs do V1).

## Mudanças V2 vs V1 (resumo)

| Hiperparâmetro | V1 | **V2** | Por quê |
|---|---|---|---|
| LR | 1e-4 uniforme | **backbone 1e-5, head 1e-4** | head reinicializado precisa aprender mais rápido que backbone pretrained |
| `drop_rate` | 0.0 | **0.3** | dropout no classifier head — regulariza a parte com pesos aleatórios |
| `weight_decay` | 0.05 | **0.1** | combate memorização |
| Augmentation | leve (±5°, jitter 0.1) | **forte (±15°, jitter 0.2, translation 5%)** | aumenta variação por epoch |
| `warmup_epochs` | 1 | **2** | dá tempo de aquecer LR diferenciado |
| `early_stopping_patience` | 3 | **5** | LR menor → convergência mais lenta |

**Não muda**: split, dataset, classes, métricas, weighted loss, image_size, batch_size, optimizer, scheduler.

## Por que rodar no Kaggle

[F-0011](https://github.com/ThiagoCB1900/TCC/blob/main/docs/findings/0011-drive-symlink-i-o-bottleneck-no-colab.md) — Colab+Drive tem gargalo de I/O. Kaggle tem o dataset (`ninadaithal/imagesoasis`) em SSD local — sem gargalo ([F-0012](https://github.com/ThiagoCB1900/TCC/blob/main/docs/findings/0012-kaggle-notebooks-resolve-i-o-do-drive.md)).

## Pré-requisitos (uma vez por conta Kaggle)

1. Conta Kaggle (gratuita, com email).
2. **Adicionar o dataset ao notebook**: painel direito → **+ Add Input** → **Search Datasets** → `ninadaithal/imagesoasis` → **Add**. Vai aparecer em `/kaggle/input/imagesoasis/`.
3. **Ativar GPU**: painel direito → **Settings** → **Accelerator** → **GPU T4 x1** (ou P100). Salvar.
4. **Ativar Internet**: painel direito → **Settings** → **Internet** → **On** (necessário pra `git clone`).
5. Execute célula por célula (recomendado para acompanhar).

## Outputs

Tudo gera em `/kaggle/working/runs/<run_id>/`. Ao final, há uma célula para compactar tudo num zip e baixar para commitar no repo (history.json, config.yaml, final_test_metrics.json). Checkpoints `.pt` ficam fora do git.

## Estimativa de tempo

V2 com LR menor pode rodar mais epochs antes do early stopping (era ~4 no V1, esperamos ~8-12 no V2). Cada epoch ~5-7 min em T4. **Treino V2 com peso: ~1-1,5h**; ablação V2 sem peso: similar. Total: ~2-3h.

As runs V1 anteriores (já em `experiments/runs/` localmente) serão usadas no comparativo da célula 13.

## 1. Verificar GPU e ambiente

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}  (CUDA: {torch.cuda.is_available()})')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  | VRAM: {p.total_memory/1e9:.1f} GB')
assert torch.cuda.is_available(), 'GPU nao detectada. Settings -> Accelerator -> GPU.'

## 2. Verificar dataset Kaggle

O Kaggle pode montar o dataset em caminhos diferentes (`/kaggle/input/imagesoasis`, `/kaggle/input/datasets/ninadaithal/imagesoasis`, com ou sem subpasta `Data/`). A célula abaixo **procura automaticamente** a pasta `Non Demented` sob `/kaggle/input` e deriva `DATA_DIR` dela — não depende do caminho exato. Em seguida valida a contagem por classe contra a EDA (F-0001/F-0002).

In [ ]:
import os, glob

# Detecção robusta do dataset: o Kaggle pode montar em caminhos diferentes
# (/kaggle/input/imagesoasis, /kaggle/input/datasets/ninadaithal/imagesoasis,
#  com ou sem subpasta Data/). Procuramos a pasta "Non Demented" em qualquer
# lugar sob /kaggle/input e derivamos DATA_DIR dela.
hits = glob.glob('/kaggle/input/**/Non Demented', recursive=True)
assert hits, (
    'Nao encontrei a pasta "Non Demented" sob /kaggle/input. '
    'Adicione o dataset via painel direito: + Add Input -> ninadaithal/imagesoasis.'
)
DATA_DIR = os.path.dirname(hits[0])
print(f'Data dir detectado: {DATA_DIR}')

EXPECTED = {
    'Non Demented': 67222,
    'Very mild Dementia': 13725,
    'Mild Dementia': 5002,
    'Moderate Dementia': 488,
}
import re
PATTERN = re.compile(r'^OAS1_\d{4}_MR[12]_mpr-[1-6]_\d{3}\.jpg$')

mismatch = False
for cls, expected in EXPECTED.items():
    cls_dir = os.path.join(DATA_DIR, cls)
    if not os.path.isdir(cls_dir):
        print(f'  {cls:25s} AUSENTE em {DATA_DIR}')
        mismatch = True
        continue
    files = os.listdir(cls_dir)
    matched = sum(1 for f in files if PATTERN.match(f))
    status = 'OK' if matched == expected else f'!! esperado {expected}'
    print(f'  {cls:25s} total={len(files):>6d} no_padrao={matched:>6d}  {status}')
    if matched != expected:
        mismatch = True

if mismatch:
    print('\n[AVISO] Contagens nao batem com manifest.csv local. Pode ter sido atualizacao do dataset Kaggle.')
    print('Se necessario, re-gerar manifest+split antes do treino (rodar src.data.eda e src.data.splits).')
else:
    print('\nOK: dataset Kaggle bate exatamente com nossa EDA (86.437 slices).')

## 3. Clonar repositório

Repo público em `github.com/ThiagoCB1900/TCC`. Internet precisa estar **On** (Settings → Internet).

In [ ]:
%cd /kaggle/working
if not os.path.isdir('/kaggle/working/TCC'):
    !git clone https://github.com/ThiagoCB1900/TCC.git
else:
    %cd /kaggle/working/TCC
    !git fetch --all && git reset --hard origin/main
%cd /kaggle/working/TCC
!git log --oneline -3

## 4. Ligar dataset ao layout do projeto

O `manifest.csv` tem paths tipo `Data/Non Demented/OAS1_xxx.jpg`. Para o `OASISDataset` resolver corretamente com `data_root='.'`, criamos symlink `/kaggle/working/TCC/Data → DATA_DIR`.

In [ ]:
import os
if os.path.islink('/kaggle/working/TCC/Data') or os.path.exists('/kaggle/working/TCC/Data'):
    os.system('rm -rf /kaggle/working/TCC/Data')
os.symlink(DATA_DIR, '/kaggle/working/TCC/Data')
!ls -la /kaggle/working/TCC/Data | head -5

## 5. Outputs persistentes (`/kaggle/working/runs/`)

Como `/kaggle/working/` persiste no notebook salvo, deixamos `experiments/runs/` apontando pra lá. Assim os artefatos das runs ficam disponíveis na versão do notebook (sem precisar de Drive).

In [ ]:
os.makedirs('/kaggle/working/runs', exist_ok=True)
if os.path.islink('/kaggle/working/TCC/experiments/runs') or os.path.exists('/kaggle/working/TCC/experiments/runs'):
    os.system('rm -rf /kaggle/working/TCC/experiments/runs')
os.symlink('/kaggle/working/runs', '/kaggle/working/TCC/experiments/runs')
!ls -la /kaggle/working/TCC/experiments/

## 6. Instalar dependências (deltas)

PyTorch + CUDA já vêm no Kaggle. Instala só o que falta.

In [ ]:
!pip install -q timm==1.0.11 monai==1.3.2 torchmetrics==1.4.3 tabulate==0.9.0 pypdf==5.0.0 seaborn==0.13.2

In [ ]:
import torch, timm, monai, torchmetrics
print(f'torch       : {torch.__version__}  CUDA={torch.cuda.is_available()}')
print(f'timm        : {timm.__version__}')
print(f'monai       : {monai.__version__}')
print(f'torchmetrics: {torchmetrics.__version__}')

## 7. Sanity do `OASISDataset`

Mesmas asserções da CPU local: 60.329 slices train / 242 pacientes / `class_to_idx` ordenado por severidade.

In [ ]:
%cd /kaggle/working/TCC
import sys
if '/kaggle/working/TCC' not in sys.path:
    sys.path.insert(0, '/kaggle/working/TCC')
from src.data.dataset import OASISDataset

train_ds = OASISDataset(
    manifest_path='results/eda/manifest.csv',
    split_path='experiments/splits/split_v1.json',
    fold='train', label_scheme='class_3', data_root='.',
)
print(f'train: {len(train_ds)} slices, {train_ds.subject_counts()} pacientes')
print(f'class_to_idx: {train_ds.class_to_idx}')
print(f'class_counts: {train_ds.class_counts()}')

## 8. Smoke test (~2 min)

Validação rápida com GPU. Esperado: loss diminui, matriz de confusão tem amostras das 3 classes (via `shuffle_eval=True` do smoke — F-0010).

In [ ]:
!python -m src.training.run \
    --smoke --smoke-batches 30 \
    --batch-size 32 --batch-size-eval 64 \
    --num-workers 2 \
    --run-name resnet50_smoke_kaggle

## 9. Treino V2 — baseline com weighted CE (ADR-0007 + ADR-0010)

Defaults V2 (já são os defaults do `src.training.run`):
- `--lr-head 1e-4 --lr-backbone 1e-5` (LR diferenciado)
- `--drop-rate 0.3` (dropout no head)
- `--weight-decay 0.1`, `--warmup-epochs 2`
- `--augment strong` (rotação ±15°, jitter 0.2, translation 5%)
- `--patience 5` (com early stopping em `balanced_accuracy`)

Para reproduzir o **V1** (comparativo) bastaria adicionar `--uniform-lr --lr 1e-4 --weight-decay 0.05 --warmup-epochs 1 --drop-rate 0.0 --augment light --patience 3`. Já temos esses resultados — não precisamos repetir.

In [ ]:
!python -m src.training.run \
    --epochs 20 \
    --batch-size 32 --batch-size-eval 64 \
    --num-workers 2 \
    --run-name resnet50_v2_class3_weighted_kaggle

## 10. Análise — curvas de treino e matriz de confusão

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

runs_dir = Path('/kaggle/working/runs')
runs = sorted([d for d in runs_dir.iterdir() if d.is_dir() and 'v2_class3_weighted_kaggle' in d.name and 'smoke' not in d.name])
assert runs, 'Nenhuma run v2_weighted_kaggle ainda.'
run = runs[-1]
print(f'Analisando: {run.name}')

history = json.loads((run / 'history.json').read_text(encoding='utf-8'))
final = json.loads((run / 'final_test_metrics.json').read_text(encoding='utf-8'))

epochs = [h['epoch'] for h in history]
tr = [h['train_loss'] for h in history]
vl = [h['val_loss'] for h in history]
ba = [h['val_metrics']['balanced_accuracy'] for h in history]
mf = [h['val_metrics']['macro_f1'] for h in history]
au = [h['val_metrics']['auc_macro'] or 0 for h in history]

fig, ax = plt.subplots(2, 2, figsize=(13, 9))
ax[0,0].plot(epochs, tr, 'o-', label='train'); ax[0,0].plot(epochs, vl, 's-', label='val')
ax[0,0].set_title('Loss V2'); ax[0,0].set_xlabel('epoca'); ax[0,0].legend()
ax[0,1].plot(epochs, ba, 'o-', color='C2', label='balanced acc'); ax[0,1].plot(epochs, mf, 's-', color='C3', label='macro F1')
ax[0,1].set_title('Metricas balanceadas V2 (val)'); ax[0,1].set_xlabel('epoca'); ax[0,1].legend(); ax[0,1].set_ylim(0,1)
ax[1,0].plot(epochs, au, 'o-', color='C4'); ax[1,0].set_title('AUC macro V2 (val)'); ax[1,0].set_xlabel('epoca'); ax[1,0].set_ylim(0.5,1)
cm = np.array(final['test_metrics']['confusion'])
cn = final['test_metrics']['class_names']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cn, yticklabels=cn, ax=ax[1,1])
ax[1,1].set_title('Matriz confusao V2 (test)'); ax[1,1].set_xlabel('predito'); ax[1,1].set_ylabel('real')
plt.tight_layout(); plt.savefig(run / 'final_curves.png', dpi=130); plt.show()

tm = final['test_metrics']
print('\n=== TEST METRICS V2 (com peso) ===')
for k in ['accuracy', 'balanced_accuracy', 'macro_f1', 'auc_macro']:
    print(f'  {k:20s}: {tm[k]:.4f}')
for cls in cn:
    p = tm['precision_per_class'][cls]; r = tm['recall_per_class'][cls]; f = tm['f1_per_class'][cls]
    print(f'  {cls:20s} P={p:.3f} R={r:.3f} F1={f:.3f}')

## 11. Ablação obrigatória — treino sem peso (ADR-0007)

Mesmo seed, mesmo split — só muda a loss. Comparativo na próxima célula.

In [ ]:
!python -m src.training.run \
    --epochs 20 \
    --batch-size 32 --batch-size-eval 64 \
    --num-workers 2 \
    --no-class-weights \
    --run-name resnet50_v2_class3_noweight_kaggle

In [ ]:
# Comparativo V1 vs V2 — busca todas as runs (locais + kaggle) e mostra lado a lado.
# Não-Kaggle: se você tem as runs V1 commitadas no repo, elas aparecem aqui automaticamente.
import pandas as pd

CATEGORIES = [
    ('V1 weighted', lambda n: 'class3_weighted_kaggle' in n and 'v2' not in n),
    ('V1 noweight', lambda n: 'class3_noweight_kaggle' in n and 'v2' not in n),
    ('V2 weighted', lambda n: 'v2_class3_weighted_kaggle' in n),
    ('V2 noweight', lambda n: 'v2_class3_noweight_kaggle' in n),
]

rows = []
for label, pred in CATEGORIES:
    matching = sorted([d for d in runs_dir.iterdir() if d.is_dir() and pred(d.name) and 'smoke' not in d.name])
    if not matching:
        print(f'(sem run "{label}")'); continue
    r = matching[-1]
    f = json.loads((r / 'final_test_metrics.json').read_text(encoding='utf-8'))['test_metrics']
    rows.append({
        'config': label,
        'run': r.name,
        'accuracy': f['accuracy'],
        'balanced_accuracy': f['balanced_accuracy'],
        'macro_f1': f['macro_f1'],
        'auc_macro': f['auc_macro'],
        'F1_non': f['f1_per_class'].get('non_demented'),
        'F1_very_mild': f['f1_per_class'].get('very_mild'),
        'F1_mild_or_mod': f['f1_per_class'].get('mild_or_moderate'),
    })
df = pd.DataFrame(rows).set_index('config')
print(df.T.to_string())
df

## 12. Compactar artefatos para baixar e commitar

Gera `/kaggle/working/runs_to_commit.zip` com os JSONs (history, config, métricas finais) e a figura final de cada run. **Pesos `.pt` ficam fora** (são grandes e ignorados no `.gitignore`).

In [ ]:
import shutil
from pathlib import Path

out = Path('/kaggle/working/runs_to_commit')
if out.exists():
    shutil.rmtree(out)
out.mkdir(parents=True)

for run in runs_dir.iterdir():
    if not run.is_dir() or 'smoke' in run.name:
        continue
    target = out / run.name
    target.mkdir()
    for fname in ['config.yaml', 'history.json', 'final_test_metrics.json', 'train.log', 'final_curves.png']:
        src = run / fname
        if src.exists():
            shutil.copy(src, target / fname)

shutil.make_archive('/kaggle/working/runs_to_commit', 'zip', out)
print('Arquivo pronto: /kaggle/working/runs_to_commit.zip')
print('Baixe-o pelo painel direito (File browser) e descompacte em experiments/runs/ local. Depois git add + commit + push.')
!ls -la /kaggle/working/runs_to_commit.zip

## 13. Próximos passos

1. **Salvar versão do notebook** (canto superior direito → **Save version** → **Save & Run All** ou apenas **Quick Save**) — preserva inputs+código+outputs como snapshot.
2. **Baixar `runs_to_commit.zip`**, descompactar localmente em `experiments/runs/`, e commitar com mensagem tipo `resultados ResNet-50 baseline (kaggle T4, com vs sem peso)`.
3. **Anotar os números** no documento do TCC (planilha sugerida no CLAUDE.md).
4. **Mostrar pro orientador** os dois resultados (com peso vs sem peso) — comparativo direto valida ADR-0007.
5. **Próxima fase:** implementar ViT-Base/16 (`src/models/vit.py` + `build_vit_base16(...)`) seguindo o mesmo contrato; este notebook serve de template (trocar `resnet50` por `vit_base_16` nos `--run-name` + nova célula de modelo).